# 📱 Google Drive Cloud Unzipper (Engine & Android Bridge)
### Extract Gigabyte-sized ZIP / RAR / 7Z files inside Google Drive in seconds without downloading anything to your phone!

**Features:**
- ⚡ **Zero Mobile Data & Zero Phone Storage:** All extraction happens directly inside Google Cloud.
- 🏎️ **Ultra-Fast 7-Zip Engine:** Multi-core extraction for huge 10GB-30GB video archives.
- 📱 **Android App Connected:** Control everything from your native Android phone app.

## 🚀 1. Launch Cloud Worker for Android App
Run this cell. It will connect your Google Drive, start the high-speed extraction engine, and keep the link active for your Android app.

In [ ]:
# Optional: Set a permanent Cloudflare Tunnel Token if you have a Cloudflare Zero Trust tunnel
# Leave empty for standard quick tunnel
CLOUDFLARE_TUNNEL_TOKEN = ""
#@title 🚀 1-CLICK LAUNCH CLOUD WORKER FOR ANDROID APP
import os, sys, time, re, threading, subprocess
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    print("🔄 Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted.")

# 2. Install tools
print("📦 Installing high-speed 7-Zip & backend packages...")
subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full", "unrar-free", "aria2"], check=False)
subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn", "pydantic"], check=False)

# 3. Download Cloudflare Tunnel
if not os.path.exists('/content/cloudflared'):
    print("🌐 Setting up Cloudflare tunnel...")
    subprocess.run(["curl", "-s", "-L", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", "/content/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

# 4. Define FastAPI app in memory
import asyncio, glob, json
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import uvicorn

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class TransferRequest(BaseModel):
    url: str
    destination_folder: str = "GDFlix"
    custom_filename: str = ""
    auto_extract: bool = True
    extract_destination: str = "MOVIES & WEB SERIES INFO"

class ExtractRequest(BaseModel):
    source_folder: str = "GDFlix"
    destination_folder: str = "MOVIES & WEB SERIES INFO"
    exact_file_name: str = ""
    password: str = ""

@app.get("/")
def health():
    return {"status": "online", "drive": os.path.exists("/content/drive/MyDrive")}

@app.post("/api/extract-stream")
async def extract_stream(req: ExtractRequest):
    async def event_generator():
        yield f"data: {json.dumps({'status': 'starting', 'message': '🚀 Initializing Cloud Extraction...'})}\\n\\n"
        drive_base = "/content/drive/MyDrive"
        src_path = os.path.join(drive_base, req.source_folder.strip("/"))
        dest_path = os.path.join(drive_base, req.destination_folder.strip("/"))

        yield f"data: {json.dumps({'status': 'scanning', 'message': f'🔍 Scanning: {src_path}'})}\\n\\n"
        if not os.path.exists(src_path):
            yield f"data: {json.dumps({'status': 'error', 'message': f'❌ Source folder not found: {src_path}'})}\\n\\n"
            return

        target_archive = None
        if req.exact_file_name.strip():
            candidate = os.path.join(src_path, req.exact_file_name.strip())
            if os.path.exists(candidate):
                target_archive = candidate
            else:
                search_term = req.exact_file_name.strip().lower()
                for f in os.listdir(src_path):
                    if search_term in f.lower() or f.lower() in search_term:
                        target_archive = os.path.join(src_path, f)
                        break

        if not target_archive:
            files = []
            for ext in ('*.zip', '*.rar', '*.7z', '*.tar', '*.tgz'):
                files.extend(glob.glob(os.path.join(src_path, ext)))
                files.extend(glob.glob(os.path.join(src_path, ext.upper())))
            if files:
                files.sort(key=os.path.getmtime, reverse=True)
                target_archive = files[0]

        if not target_archive or not os.path.exists(target_archive):
            yield f"data: {json.dumps({'status': 'error', 'message': f'❌ No archive found in {req.source_folder}'})}\\n\\n"
            return

        os.makedirs(dest_path, exist_ok=True)
        size_gb = os.path.getsize(target_archive) / (1024 ** 3)
        fname = os.path.basename(target_archive)
        yield f"data: {json.dumps({'status': 'progress', 'progress': 10, 'message': f'🎯 Target: {fname} ({size_gb:.2f} GB)'})}\\n\\n"
        yield f"data: {json.dumps({'status': 'progress', 'progress': 20, 'message': f'📂 Extracting to: MyDrive/{req.destination_folder}'})}\\n\\n"

        cmd = ["7z", "x", "-y", "-bsp1", "-mmt=on", f"-o{dest_path}"]
        if req.password: cmd.append(f"-p{req.password}")
        else: cmd.append("-p-")
        cmd.append(target_archive)

        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        last_pct = 20
        buf = ""
        while True:
            char = proc.stdout.read(1)
            if not char and proc.poll() is not None:
                break
            if char:
                buf += char
                if char in ('\r', '\n'):
                    l = buf.strip()
                    buf = ""
                    if l:
                        m_pct = re.search(r'(\d+)%', l)
                        if m_pct:
                            pct_val = int(m_pct.group(1))
                            mapped = 20 + int(pct_val * 0.78)
                            if mapped > last_pct:
                                last_pct = mapped
                            payload = {'status': 'progress', 'progress': last_pct, 'message': f'⏳ Extracting: {pct_val}% ({l[:60]})'}
                            yield f"data: {json.dumps(payload)}\\n\\n"
                            await asyncio.sleep(0.01)
                        elif "Extracting" in l:
                            payload = {'status': 'progress', 'progress': last_pct, 'message': f'⏳ {l[:80]}'}
                            yield f"data: {json.dumps(payload)}\\n\\n"
                            await asyncio.sleep(0.01)
        proc.wait()

        if proc.returncode == 0:
            yield f"data: {json.dumps({'status': 'success', 'progress': 100, 'message': f'🎉 ✅ SUCCESS! All files extracted to MyDrive/{req.destination_folder}'})}\\n\\n"
        else:
            yield f"data: {json.dumps({'status': 'error', 'message': f'❌ Extraction failed with code {proc.returncode}'})}\\n\\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")

@app.post("/api/transfer-stream")
async def transfer_stream(req: TransferRequest):
    async def event_generator():
        yield f"data: {json.dumps({'status': 'starting', 'message': '🚀 Initializing Cloud Transfer Engine...'})}\n\n"
        drive_base = "/content/drive/MyDrive"
        if not os.path.exists(drive_base):
            yield f"data: {json.dumps({'status': 'error', 'message': '❌ Drive not mounted at /content/drive/MyDrive'})}\n\n"
            return

        target_url = req.url.strip()
        if not target_url or not target_url.startswith(("http://", "https://")):
            yield f"data: {json.dumps({'status': 'error', 'message': '❌ Invalid download URL provided.'})}\n\n"
            return

        dest_dir = os.path.join(drive_base, req.destination_folder.strip("/"))
        os.makedirs(dest_dir, exist_ok=True)

        yield f"data: {json.dumps({'status': 'progress', 'progress': 10, 'message': f'📂 Destination: MyDrive/{req.destination_folder}'})}\n\n"

        # Smart GDFlix URL resolving
        if "gdflix.io/file/" in target_url:
            wurl = target_url.replace("/file/", "/wfile/")
            try:
                import urllib.request
                h_req = urllib.request.Request(wurl, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(h_req, timeout=5) as u_resp:
                    html_c = u_resp.read().decode('utf-8', errors='ignore')
                    found = re.findall(r'href=["'](https?://[^'"]*(?:workers\.dev|gofile\.io)[^'"]*)["']', html_c)
                    if found:
                        target_url = found[0]
                        yield f"data: {json.dumps({'status': 'progress', 'progress': 12, 'message': '✅ Resolved direct cloud link!'})}\n\n"
            except Exception:
                pass

        yield f"data: {json.dumps({'status': 'progress', 'progress': 15, 'message': '⚡ Engine: Multi-stream aria2c accelerator (16 connections)'})}\n\n"

        cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--file-allocation=none", "--summary-interval=1", "--console-log-level=warn", "--allow-overwrite=true", "-d", dest_dir]
        if req.custom_filename.strip():
            cmd.extend(["-o", req.custom_filename.strip()])
        cmd.append(target_url)

        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        last_pct = 15
        aria_regex = re.compile(r'\[#[0-9a-fA-F]+\s+([0-9.]+\w+)/([0-9.]+\w+)\(([0-9]+)%\)\s+CN:\d+\s+DL:([0-9.]+\w+)(?:\s+ETA:([0-9a-zA-Z]+))?\]')

        buf = ""
        while True:
            char = proc.stdout.read(1)
            if not char and proc.poll() is not None:
                break
            if char:
                buf += char
                if char in ('\r', '\n'):
                    line = buf.strip()
                    buf = ""
                    if line:
                        m = aria_regex.search(line)
                        if m:
                            downloaded, total, pct_str, speed, eta = m.groups()
                            pct = int(pct_str)
                            if pct > last_pct:
                                last_pct = pct
                            eta_str = f" • ETA: {eta}" if eta else ""
                            yield f"data: {json.dumps({'status': 'progress', 'progress': last_pct, 'message': f'📥 {downloaded}/{total} ({pct}%) • {speed}/s{eta_str}'})}\n\n"
                            await asyncio.sleep(0.01)
        proc.wait()

        if proc.returncode != 0:
            yield f"data: {json.dumps({'status': 'error', 'message': '❌ Download failed. Check direct link accessibility.'})}\n\n"
            return

        candidates = [os.path.join(dest_dir, f) for f in os.listdir(dest_dir)]
        candidates.sort(key=os.path.getmtime, reverse=True)
        saved_file = candidates[0] if candidates else None
        fname = os.path.basename(saved_file) if saved_file else "file"
        size_gb = (os.path.getsize(saved_file) / (1024**3)) if saved_file else 0.0

        yield f"data: {json.dumps({'status': 'progress', 'progress': 100, 'message': f'💾 ✅ Saved: {fname} ({size_gb:.2f} GB) in MyDrive/{req.destination_folder}'})}\n\n"

        if req.auto_extract and saved_file and saved_file.endswith(('.zip', '.rar', '.7z', '.tar', '.tgz')):
            yield f"data: {json.dumps({'status': 'progress', 'progress': 10, 'message': f'🚀 Auto-Extracting {fname} to MyDrive/{req.extract_destination}...'})}\n\n"
            extract_dest = os.path.join(drive_base, req.extract_destination.strip("/"))
            os.makedirs(extract_dest, exist_ok=True)
            cmd_ext = ["7z", "x", "-y", "-bsp1", "-mmt=on", f"-o{extract_dest}", "-p-", saved_file]
            proc_ext = subprocess.Popen(cmd_ext, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            buf_e = ""
            while True:
                c = proc_ext.stdout.read(1)
                if not c and proc_ext.poll() is not None:
                    break
                if c:
                    buf_e += c
                    if c in ('\r', '\n'):
                        le = buf_e.strip()
                        buf_e = ""
                        if le:
                            me = re.search(r'(\d+)%', le)
                            if me:
                                pct_ext = int(me.group(1))
                                yield f"data: {json.dumps({'status': 'progress', 'progress': pct_ext, 'message': f'⏳ Extracting: {pct_ext}% ({le[:50]})'})}\n\n"
                                await asyncio.sleep(0.01)
            proc_ext.wait()
            if proc_ext.returncode == 0:
                yield f"data: {json.dumps({'status': 'success', 'progress': 100, 'message': f'🎉 ✅ SUCCESS! Transferred & Extracted to MyDrive/{req.extract_destination}'})}\n\n"
            else:
                yield f"data: {json.dumps({'status': 'error', 'message': f'⚠️ Downloaded, but extraction exited with code {proc_ext.returncode}'})}\n\n"
        else:
            yield f"data: {json.dumps({'status': 'success', 'progress': 100, 'message': f'🎉 ✅ SUCCESS! Transferred to MyDrive/{req.destination_folder}'})}\n\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")

# 5. Start Server in In-Memory Thread
print("⚡ Starting in-process API server...")
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True
)
server_thread.start()
time.sleep(1.5)

# Verify local server response
import urllib.request
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/") as resp:
        print("✅ Local Server confirmed running (HTTP 200 OK)!")
except Exception as e:
    print(f"⚠️ Warning: {e}")

# 6. Start Cloudflare Tunnel
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
if os.path.exists("/content/tunnel.log"):
    os.remove("/content/tunnel.log")

tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", "/content/tunnel.log"]
)

print("⏳ Establishing secure link to your Android phone...")
tunnel_url = None
for _ in range(60):
    time.sleep(0.5)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as lf:
            content = lf.read()
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    print("\n" + "=" * 65)
    print("🎉 CLOUD WORKER IS ONLINE & READY!")
    print("=" * 65)
    print(f"\n📱 COPY & PASTE THIS URL INTO YOUR ANDROID APP:\n")
    print(f"👉  {tunnel_url}  👈\n")
    print("=" * 65)
    print("🟢 WORKER ACTIVE! Keep this cell running while extracting from your phone.")
    print("To stop later, click the stop button (■) on this cell.\n")
    try:
        while True:
            time.sleep(5)
    except KeyboardInterrupt:
        print("\n⏹️ Worker stopped.")
else:
    print("❌ Could not detect Cloudflare URL yet. Check /content/tunnel.log")
